LoRa fine-tuning для генерации стихотворений в стиле Маяковского

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np



In [4]:
dataset = load_dataset("AnyaSchen/russian_poetry_with_keywords", split="train")
df = dataset.to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/442 [00:00<?, ?B/s]

data/train-00000-of-00001-a239d0191cfafa(…):   0%|          | 0.00/2.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7755 [00:00<?, ? examples/s]

In [5]:
data = df[df["author"] == "Маяковский"]
data

,text,author,keywords
0,"Влас Прогулкин —\nмилый мальчик,\nспать ложилс...",Маяковский,"['спать', 'журнальчик', 'заставить', 'мальчик'..."
1,"Засыпает на рассвете,\nскомкав\nёрзаньем\nкров...",Маяковский,"['вставать', 'детвора', 'отец', 'засыпать', 'у..."
2,"Разошлись\nдругие\nв школы,–\nВлас\nу крана\nп...",Маяковский,"['мочить', 'дрематься', 'выходить', 'школа', '..."
3,Пошагал\nи встал разиней:\nвывеска на магазине...,Маяковский,"['магазин', 'вывеска', 'прочесть', 'пошагать',..."
4,"С конца прочёл\nзнаток наук, —\nНомисвыходит\n...",Маяковский,"['номисвыходить', 'наука', 'пять', 'прочесть',..."
...,...,...,...
1102,Наши ткани\n не богатым —\nобошьешь...,Маяковский,"['магазин', 'зарплата', 'богатый', 'ткань', 'в..."
1103,Так уверенно глядели\nГаша с пряхой в женотдел...,Маяковский,"['женотдел', 'подружка', 'петрушкин', 'читать'..."
1104,"Старины развейтесь тени,\nне гнусите под окном...",Маяковский,"['старина', 'одеть', 'фабрика', 'развеяться', ..."
1105,Пред зовет завхоза басом:\n«Вдрызг конец прише...,Маяковский,"['писчебумажность', 'канцелярия', 'прийти', 'с..."


In [6]:
data["keywords"][1]

"['вставать', 'детвора', 'отец', 'засыпать', 'ухо']"

In [7]:
data["text"][1]

'Засыпает на рассвете,\nскомкав\nёрзаньем\nкровать,\nв час,\nкогда\nдругие дети\nбодро\nначали вставать.\nКогда\nдругая детвора\nчаёвничает, вставши,\nотец\nорёт ему:\n– Пора!–\nОн —\nодеяло на уши.\n'

In [8]:
def format_poem_for_chat(row):
    """Форматирует строку DataFrame в диалоговый формат"""

    # Очистка текста
    poem_cleaned = str(row["text"]).replace("\n\n", "\n").strip()

    # Обработка ключевых слов
    keywords = str(row["keywords"])

    # Создаем диалог
    formatted_text = f"""<|system|>
Ты — поэт, который пишет в стиле Маяковского. Твои стихи энергичные, с резкими ритмами и необычными образами.
<|user|>
Напиши стихотворение в стиле Маяковского, используя слова: {keywords}.
<|assistant|>
{poem_cleaned}
</s>"""

    return formatted_text

formatted = format_poem_for_chat(data.iloc[0])
print(formatted)

<|system|>
Ты — поэт, который пишет в стиле Маяковского. Твои стихи энергичные, с резкими ритмами и необычными образами.
<|user|>
Напиши стихотворение в стиле Маяковского, используя слова: ['спать', 'журнальчик', 'заставить', 'мальчик', 'мать'].
<|assistant|>
Влас Прогулкин —
милый мальчик,
спать ложился,
взяв журнальчик.
Все в журнале
интересно.
– Дочитаю весь,
хоть тресну!
– Ни отец его,
ни мать
не могли
заставить спать.
</s>


In [9]:
formatted_poems = data.apply(format_poem_for_chat, axis=1)
dataset = pd.DataFrame({"text": formatted_poems})

In [10]:
dataset

,text
0,"<|system|>\nТы — поэт, который пишет в стиле М..."
1,"<|system|>\nТы — поэт, который пишет в стиле М..."
2,"<|system|>\nТы — поэт, который пишет в стиле М..."
3,"<|system|>\nТы — поэт, который пишет в стиле М..."
4,"<|system|>\nТы — поэт, который пишет в стиле М..."
...,...
1102,"<|system|>\nТы — поэт, который пишет в стиле М..."
1103,"<|system|>\nТы — поэт, который пишет в стиле М..."
1104,"<|system|>\nТы — поэт, который пишет в стиле М..."
1105,"<|system|>\nТы — поэт, который пишет в стиле М..."


In [11]:

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"


tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()



tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [12]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

hf_dataset = Dataset.from_pandas(dataset)
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1107 [00:00<?, ? examples/s]

In [13]:
tokenized_dataset

Dataset({
    features: ['text', '__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1107
})

In [16]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.7 MB/s eta 0:00:00


In [14]:
split_dataset = tokenized_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [ ]:
def preprocess_dataset(dataset):
    """Удаляем лишние колонки из датасета"""
    # Оставляем только нужные колонки
    dataset = dataset.remove_columns(
        [col for col in dataset.column_names
         if col not in ["input_ids", "attention_mask", "labels"]]
    )
    return dataset

train_dataset = preprocess_dataset(train_dataset)
eval_dataset = preprocess_dataset(eval_dataset)

In [16]:
training_args = TrainingArguments(
    output_dir="./tinyllama-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_first_step=False,
    logging_nan_inf_filter=False,
)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [13]:
# training_args = TrainingArguments(
#     output_dir="./tinyllama-finetuned",
#     num_train_epochs=3,
#     per_device_train_batch_size=5,
#     gradient_accumulation_steps=4,
#     warmup_steps=10,
#     logging_steps=5,
#     save_strategy="no",
#     learning_rate=2e-4,
#     fp16=False,
#     optim="adamw_torch",
#     report_to="none",
# )

# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer,
#     mlm=False
# )

In [17]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)
trainer.train()

lora_model.save_pretrained("./tinyllama-diet-lora")
print("✓ LoRA веса сохранены в ./tinyllama-diet-lora")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss
1,1.889100,1.536853
2,1.464200,1.478303
3,1.419700,1.460611
4,1.395400,1.452030
5,1.375000,1.447996
6,1.359200,1.444796
7,1.345700,1.444753
8,1.332900,1.444357
9,1.326000,1.444687
10,1.319500,1.445200


✓ LoRA веса сохранены в ./tinyllama-diet-lora


In [18]:
def generate_response(prompt, model, tokenizer, max_length=300):
    formatted_prompt = f"""<|system|>
Ты — поэт, который пишет в стиле Маяковского. Твои стихи энергичные, с резкими ритмами и необычными образами.
<|user|>
Напиши стихотворение в стиле Маяковского, используя слова: {prompt}.
<|assistant|>
"""

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response_start = full_text.find("<|assistant|>") + len("<|assistant|>")
    response = full_text[response_start:].strip()

    response = response.replace("</s>", "").strip()

    return response

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
).eval()

test_queries = [
    "рассвет, метро, толпа",
    "любовь, революция, улица",
    "осень, дождь, одиночество",
    "мечта, звёзды, высота",
    "город, небо, свет"
]

lora_model.eval()
base_model.eval()

results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"ТЕСТ {i}: Ключевые слова: {query}")
    print(f"{'='*60}")

    print("\nБазовая модель:")
    base_response = generate_response(query, base_model, tokenizer)
    print(base_response)

    print("\nМодель после fine-tuning:")
    our_response = generate_response(query, lora_model, tokenizer)
    print(our_response)

    results.append({
        "query": query,
        "base_response": base_response,
        "our_response": our_response
    })


ТЕСТ 1: Ключевые слова: рассвет, метро, толпа

Базовая модель:
Дорогая любовница,
Выберите прическу,
Солнце всегда будет нам радуть.
Рассвет, звезда, скачала нас к своим глазах.
И столбцы метро пронесли сквозь туман.
Приглашай меня в толпе,
Между мыслями у вас постоянно ходит.

Модель после fine-tuning:
Рассвета наблюдают по улицам
под закатышками в метротолпе.
Судя по всему, толпа узнала
вышеглубоко в небе рассвет.
Взлет метеорологов развалился:
всего лишь один промах.
Идет пару дней впереди всячески
наступление напоминание.
Всего лишь девять минут наступает
на заданный план рассвета.
Совершенно похожее на сказочку
наброска межводных волн.
Странное — когда звездные непропорные
в метротолпе выглядят идущими.
Чтоб это был рассвет в метротолпе
рассел квартал и поплясал пёстрый,
отсюда подъем на улицу всякий раз
раскрытый поворотный, а весь день
смешан в одном кольце со штормом.
Комментарии:
«Высокость» в метротолпе не просто
высокая.
Это

ТЕСТ 2: Ключевые слова: любовь, революция, улица